# [Generate synthetic and simulated data for evaluation](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/simulator-interaction-data) (starter)
The Azure AI Evaluation SDK `Simulator` generates synthetic conversations for testing an application before production data is available. It can use source text, predefined tasks, or custom callbacks to simulate non-adversarial interactions.

This capability is useful for:
- **Testing conversational applications:** Check how chatbots and assistants respond across representative scenarios.
- **Creating evaluation datasets:** Produce conversation records for repeatable evaluation and analysis.
- **Supporting model development:** Generate varied examples for experimentation and refinement.

This notebook connects the simulator to a Prompty-based application and produces a small grounded conversation.

In [1]:
import os, sys, json
import prompty
import asyncio
from typing import Any, Dict, Optional
from pprint import pprint
from urllib.parse import urlsplit
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv  # requires python-dotenv
from openai import OpenAI

if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

ASSETS_FOLDER = "eval_assets"
GROUNDING_DATA_SOURCE_PATH = "../data/documents.txt"
PROMPTY_APP = "conversation_simulation.prompty"
PROMPTY_CONNECTION_NAME = "workshop-foundry"

openai_api_version = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
foundry_project_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
azure_openai_deployment_name = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)

if not foundry_project_endpoint:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT must be set.")

# Convert the project endpoint to the OpenAI-compatible endpoint used by the Foundry provider.
project_host = urlsplit(foundry_project_endpoint).hostname
foundry_host_suffix = ".services.ai.azure.com"
if not project_host or not project_host.endswith(foundry_host_suffix):
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT is not a valid Azure Foundry project endpoint.")

# prompty.invoke() has no credential parameter, so register a client that uses this credential.
account_name = project_host.removesuffix(foundry_host_suffix)
foundry_openai_base_url = f"https://{account_name}.openai.azure.com/openai/v1"
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
prompty_client = OpenAI(api_key=token_provider, base_url=foundry_openai_base_url)
prompty.register_connection(PROMPTY_CONNECTION_NAME, client=prompty_client)

print(f"azure_openai_endpoint: {azure_openai_endpoint}")
print(f"foundry_project_endpoint: {foundry_project_endpoint}")
print(f"azure_openai_deployment_name: {azure_openai_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

azure_openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
foundry_project_endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project
azure_openai_deployment_name: gpt-5.4-mini
openai_api_version: 2025-04-01-preview


In [2]:
# Initialize Azure OpenAI connection

from azure.ai.evaluation import AzureOpenAIModelConfiguration

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_openai_deployment_name,
    api_version=openai_api_version,
)

## Define the Conversation Simulation Prompty
A `.prompty` file is a portable prompt asset composed of front matter that defines the model configuration and typed inputs, followed by the prompt template. The standalone `prompty` Python package loads the file, resolves the configuration and inputs, renders the template, and invokes the configured model.

Prompty is independent of Microsoft Prompt flow: using the `prompty` package does not require Prompt flow, so Prompt flow's deprecation and retirement lifecycle does not apply to this execution approach. In this example, the conversation history is declared as a `thread`, allowing Prompty to insert prior messages before the current user query.

`prompty.invoke()` does not expose a `credential` parameter. A connection declared as `kind: foundry` would create its own `DefaultAzureCredential` internally, which might select a different identity from the one configured by this notebook. The setup cell therefore creates an OpenAI-compatible client with the notebook's credential and registers it under `workshop-foundry`. The Prompty file uses `kind: reference` to retrieve that client for every invocation.

In [3]:
# Reference the credential-backed client registered in the setup cell.
with open(f"{ASSETS_FOLDER}/{PROMPTY_APP}", "w") as f:
    f.write("""---
name: ConversationSimulationPrompty
description: Chat application for simulating a conversatation
model:
    id: ${env:AZURE_OPENAI_CHAT_DEPLOYMENT_NAME}
    provider: foundry
    connection:
        kind: reference
        name: workshop-foundry
    options:
        temperature: 0.0
        top_p: 1.0
inputs:
    - name: context
      kind: string
    - name: query
      kind: string
    - name: conversation_history
      kind: thread
template:
    format:
        kind: jinja2
    parser:
        kind: prompty
---
system:
You are a helpful assistant and you're helping with the user's query.
Keep the conversation engaging and interesting.
{{ conversation_history }}

user:
Keep your answer grounded in the provided context:
{{ context }}

Continue the conversation by responding to this query:
{{ query }}""")

## Test the prompty application

In [4]:
from IPython.display import Markdown

prompty_path = f"./{ASSETS_FOLDER}/{PROMPTY_APP}"
query = ... # for example: what is a meaning function

# Keep direct Prompty execution behind a small application-style function.
def run_application(*, context: str, query: str, conversation_history: list[dict]) -> str:
    return prompty.invoke(
        prompty_path,
        inputs={
            "conversation_history": conversation_history,
            "context": context,
            "query": query,
        },
    )

# Run a simple smoke test before connecting the application to the Simulator.
Markdown(run_application(context="", query=query, conversation_history=[]))

Making pizza at home is pretty straightforward. Here’s a simple way to do it:

### Ingredients
- Pizza dough
- Tomato sauce or pizza sauce
- Mozzarella cheese
- Your favorite toppings, like pepperoni, vegetables, mushrooms, onions, or olives
- Olive oil
- Flour or cornmeal for dusting

### Steps
1. **Preheat the oven**  
   Set your oven to the highest temperature it can safely reach, usually around **475–500°F (245–260°C)**. If you have a pizza stone or baking steel, put it in the oven while it heats.

2. **Prepare the dough**  
   Let the dough come to room temperature if it’s cold. Lightly flour your surface, then stretch or roll the dough into a round shape.

3. **Add sauce and toppings**  
   Place the dough on a baking sheet, pizza pan, or preheated stone. Spread a thin layer of sauce, then add cheese and toppings. Don’t overload it, or the pizza may get soggy.

4. **Bake**  
   Bake for about **10–15 minutes**, depending on your oven and crust thickness, until the crust is golden and the cheese is bubbly.

5. **Finish and serve**  
   Let it cool for a minute or two, then slice and enjoy. You can drizzle a little olive oil or sprinkle herbs on top if you like.

If you want, I can also give you:
- a **homemade pizza dough recipe**
- a **no-yeast pizza recipe**
- or a **step-by-step recipe for a specific style** like thin crust or deep dish.

In [5]:
# friendly_conversation_history_en.txt or friendly_conversation_history_it.txt
with open(f"./{ASSETS_FOLDER}/friendly_conversation_history_it.txt", "r", encoding="utf-8") as f:
    conversation_history = json.load(f)
    
# print the conversation history
...

[{'content': 'Parliamo in italiano, con risposte su singola riga e meno di 30 parole. Cosa fai di bello questa sera?',
  'role': 'user',
  'context': 'Domanda amichevole.'},
 {'content': 'Vado al cinema',
  'role': 'assistant',
  'context': "Esposizione di un'attività ricreativa divertente."},
 {'content': 'Ah fantastico! Che cosa vai a vedere?',
  'role': 'user',
  'context': "Ulteriore domanda amichevole a dimostrare interesse verso l'interlocutore."},
 {'content': 'Lo SQUALO di Steven Spielberg.',
  'role': 'assistant',
  'context': 'Dettagli finali di questa attività divertente. Ora introduci un nuovo argomento diverso, ma connesso a questo.'}]

In [6]:
i = 0
for ch in conversation_history:
    # print message number, role and content
    ...

response = run_application(
    context=conversation_history[-1]["context"],
    query=conversation_history[-1]["content"],
    conversation_history=conversation_history[:-1],
)

# print the latest message, just returned by the last LLM invocation

Message 0 (user): Parliamo in italiano, con risposte su singola riga e meno di 30 parole. Cosa fai di bello questa sera?
Message 1 (assistant): Vado al cinema
Message 2 (user): Ah fantastico! Che cosa vai a vedere?
Message 3 (assistant): Lo SQUALO di Steven Spielberg.

Message 4 (assistant): Ottima scelta: un classico! Restando in tema cinema, ti piacciono più i thriller o i film d’avventura?


## Generate and append one additional turn to the sample conversation. 
Feel free to run the next cell multiple times, to show how the conversation cotinues

In [7]:
# Generate and append one additional turn to the sample conversation.

next_role = "user" if conversation_history[-1]["role"] == "assistant" else "assistant"
conversation_history.append({"content": response, "role": next_role, "context": ""})

for index, message in enumerate(conversation_history):
    # print message id, role and content
    ...

response = run_application(
    context=conversation_history[-1]["context"],
    query=conversation_history[-1]["content"],
    conversation_history=conversation_history[:-1],
)

# print the final answer

Message 0 (user): Parliamo in italiano, con risposte su singola riga e meno di 30 parole. Cosa fai di bello questa sera?
Message 1 (assistant): Vado al cinema
Message 2 (user): Ah fantastico! Che cosa vai a vedere?
Message 3 (assistant): Lo SQUALO di Steven Spielberg.
Message 4 (user): Ottima scelta: un classico! Restando in tema cinema, ti piacciono più i thriller o i film d’avventura?

Message 5 (assistant): Preferisco i thriller: mi tengono sempre col fiato sospeso.


## Define the Simulator callback
The callback implements the same mechanism demonstrated manually above, adapted to the interface expected by the `Simulator`. It separates the latest message from the preceding conversation history, extracts its grounding context, resolves the path to the Prompty file, and invokes it with the query, context, and history as inputs. It then appends the generated assistant response and returns the conversation using the chat protocol expected by the Evaluation SDK.

> **Prompty v2 beta compatibility note:** This workshop pins `prompty[foundry,jinja2]==2.0.0b3`, the version validated with this notebook. In this release, `prompty.invoke_async()` is not compatible with the Entra ID authentication path used by the Foundry provider: the asynchronous OpenAI client attempts to await a synchronous token provider and raises `TypeError: object str can't be used in 'await' expression`.
>
> The callback therefore runs the validated synchronous `prompty.invoke()` function through `asyncio.to_thread()`. This keeps the callback asynchronous without blocking the notebook event loop.
>
> **Simulator authentication note:** `azure-ai-evaluation==1.18.3` creates a separate legacy `AsyncPrompty` for generating simulated user turns. Its default embedded-Prompty branch does not expose or forward a credential. The simulation cell explicitly resolves the same embedded Prompty and passes `token_credential` through `user_simulator_prompty_options`, ensuring that both the user simulator and the application callback use the notebook's credential.

In [8]:
async def callback(
    messages: Dict[str, Any],
    stream: bool = False,
    session_state: Any = None,
    context: Optional[Dict[str, Any]] = None,
    assets_folder: str = ASSETS_FOLDER,
    prompty_app: str = PROMPTY_APP
) -> dict[str, Any]:
    
    messages_list = messages["messages"]
    latest_message = messages_list[-1]
    latest_context = latest_message.get("context") or ""
    application_prompty = os.path.join(os.getcwd(), assets_folder, prompty_app)

    # Run the synchronous Prompty call on a worker thread to keep this callback non-blocking.
    response = await asyncio.to_thread(
        prompty.invoke,
        application_prompty,
        inputs={
            "query": latest_message["content"],
            "context": latest_context,
            "conversation_history": messages_list[:-1],
        },
    )

    # Return the updated conversation in the protocol expected by the Simulator.
    messages_list.append(
        {
            "content": response,
            "role": "assistant",
            "context": latest_context,
        }
    )
    return {
        "messages": messages_list,
        "stream": stream,
        "session_state": session_state,
        "context": context,
    }

## Prepare the grounding text
For a deterministic workshop run, the example reads a local document instead of calling an external content API. The first 5,000 characters provide enough context for a short grounded conversation.

In [9]:
from pathlib import Path

# Limit the source text to keep the example fast and the prompt compact.
source_path = Path(GROUNDING_DATA_SOURCE_PATH)
source_text = source_path.read_text(encoding="utf-8")[:5000]
print(f"{source_text[:1000]}...")

"Bullet Kin
Bullet Kin are one of the most common enemies. They slowly walk towards the player, occasionally firing a single bullet. They can flip tables and use them as cover. They will also deal contact damage if the player touches them.

Occasionally, Bullet Kin will have assault rifles, in which case they will rapidly fire 8 bullets towards the player before reloading. When an assault rifle wielding bullet kin appears, there will often be more in the same room.

On some occasions the player will also encounter incapacitated Bullet Kin lying on the floor. These Bullet Kin are props and disintegrate upon touch. They can be found in mass quantity in Oubliette.

In the Black Powder Mine, they can also ride Minecarts. In fact, if there are any unoccupied Minecarts within the room, they will take priority by walking towards them to ride in.

Trivia
Bullet Kin wield Magnums. Assault-rifle wielding Bullet Kin wield AK-47s.
Incapacitated Bullet Kin can be found in the Oublilette and Cannon'

In [10]:
from azure.ai.evaluation.simulator import Simulator

class CompatibleSimulator(Simulator):
    """Normalize the wrapped Prompty output returned by Evaluation SDK 1.18.3."""

    def _parse_prompty_response(self, *, response: Any) -> Dict[str, Any]:
        parsed_response = super()._parse_prompty_response(response=response)
        if isinstance(parsed_response, dict) and "content" not in parsed_response:
            llm_output = parsed_response.get("llm_output")
            if isinstance(llm_output, str):
                parsed_response = super()._parse_prompty_response(response=llm_output)
            elif isinstance(llm_output, dict):
                parsed_response = llm_output
        if not isinstance(parsed_response, dict) or "content" not in parsed_response:
            raise ValueError(f"Unexpected user-simulator response: {parsed_response!r}")
        return parsed_response

# Use the compatibility subclass until the SDK unwraps llm_output internally.
simulator = CompatibleSimulator(model_config=model_config)

Class Simulator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [11]:
from importlib.resources import files

# Use the SDK's embedded user-simulator Prompty while injecting the notebook credential.
user_simulator_prompty = files("azure.ai.evaluation.simulator._prompty").joinpath("task_simulate.prompty")

# Use a predefined user turn to make the first exchange deterministic.
seed_turns = [
    [
        {
            "content": "What are Bullet Kin and how do they behave?",
            "context": source_text,
        }
    ]
]

# Each conversation turn is one user/assistant pair; the user simulator generates turns after the seed.
outputs = await simulator(
    target=callback,
    conversation_turns=seed_turns,
    max_conversation_turns=5,
    user_simulator_prompty=user_simulator_prompty,
    user_simulator_prompty_options={"token_credential": credential},
    api_call_delay_sec=0,
)

Simulating with predefined conversation turns: 100%|████████████| 5/5 [00:18<00:00,  3.68s/messages]


In [12]:
# Print the generated conversation in chronological order.
for index, message in enumerate(outputs[0]["messages"]):
    if (index==0):
        print(f"==========HERE ARE THE {len(outputs[0]["messages"])} GENERATED MESSAGES ==========\n")
    print(f'===== Message {index+1} by <{message["role"]}> =====')
    print(f'{message["content"]}\n\n')

==========HERE ARE THE 10 GENERATED MESSAGES ==========

===== Message 1 by <user> =====
What are Bullet Kin and how do they behave?


===== Message 2 by <assistant> =====
Bullet Kin are one of the most common enemies. They slowly walk toward the player and occasionally fire a single bullet. They can also flip tables to use as cover, and if the player touches them, they deal contact damage.

Sometimes they appear with assault rifles instead of their usual weapon. In that case, they rapidly fire 8 bullets toward the player before reloading, and rooms with one often contain more assault-rifle Bullet Kin.

There are also incapacitated Bullet Kin lying on the floor as props; these disintegrate when touched. In the Black Powder Mine, Bullet Kin can ride minecarts if one is available.


===== Message 3 by <user> =====
That’s helpful — do Bullet Kin have any notable weaknesses or easy ways to deal with them, especially when they’re using assault rifles or riding minecarts?


===== Message 4 b